# MB-89xx compact-model evaluation (Colab)
Runs the frozen compact checkpoints (v6-1.7B, v7-8B) on the MB-89xx
anchor + extension corpora (873 items each), archives to
`results/frontier_runs/<name>__MB-89xx.jsonl.gz`, resume-safe by
(task, sample). After both runs, score locally with
`scripts/score_mb89xx.py` (leaderboard rows) — the scorer picks the
archives up automatically.

Runtime: T4 is fine for the 1.7B (fp16); use A100 (or 4-bit load) for the 8B.



In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
%%capture
!pip install -q -U transformers peft accelerate bitsandbytes torchao

In [3]:
import gzip, json, os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

DRIVE = "/content/drive/MyDrive/masld-llm"
CORPORA = ["MB-89xx_plasmacf_test.jsonl.gz", "MB-89xx_plasmacf_ext_test.jsonl.gz"]

# ---- pick ONE configuration ----
CFG = dict(name="qwen3-1.7b-masld-v6", base="Qwen/Qwen3-1.7B",
           ckpt="checkpoints/qwen3-1.7b-masld-sft-v6", quant=False)
# CFG = dict(name="qwen3-8b-masld-v7", base="Qwen/Qwen3-8B",
#            ckpt="checkpoints/qwen3-8b-masld-sft-v7", quant=True)

OUT = f"{DRIVE}/results/frontier_runs/{CFG['name']}__MB-89xx.jsonl.gz"
print("config:", CFG, "->", OUT)


config: {'name': 'qwen3-1.7b-masld-v6', 'base': 'Qwen/Qwen3-1.7B', 'ckpt': 'checkpoints/qwen3-1.7b-masld-sft-v6', 'quant': False} -> /content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld-v6__MB-89xx.jsonl.gz


In [4]:
def load_records():
    recs = []
    for f in CORPORA:
        with gzip.open(f"{DRIVE}/prompts/{f}", "rt") as fh:
            for line in fh:
                line = line.strip()
                if not line: continue
                r = json.loads(line)
                if "__meta__" in r: continue
                recs.append(r)
    return recs

def load_done(path):
    if not os.path.exists(path): return set()
    done = set()
    try:
        with gzip.open(path, "rt") as fh:
            for line in fh:
                line = line.strip()
                if not line: continue
                r = json.loads(line)
                if r.get("task") and r.get("sample") and r.get("error") is None:
                    done.add((r["task"], r["sample"]))
    except Exception:
        pass
    return done

def append_record(path, rec):
    with gzip.open(path, "ab") as fh:
        fh.write((json.dumps(rec) + "\n").encode())


In [5]:
recs = load_records()
done = load_done(OUT)
todo = [r for r in recs if (r["task"], r["sample"]) not in done]
print(f"corpus {len(recs)} | done {len(done)} | todo {len(todo)}")

if not todo:
    print("nothing to do — archive complete")


corpus 873 | done 0 | todo 873


In [9]:
tok = AutoTokenizer.from_pretrained(CFG["base"], padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
if CFG["quant"]:
    from transformers import BitsAndBytesConfig
    base = AutoModelForCausalLM.from_pretrained(
        CFG["base"],
        quantization_config=BitsAndBytesConfig(load_in_4bit=True),
        device_map="auto"
    )
else:
    base = AutoModelForCausalLM.from_pretrained(
        CFG["base"],
        dtype=torch.float16,
        device_map="auto"
    )
model = PeftModel.from_pretrained(base, f"{DRIVE}/{CFG['ckpt']}")
model = model.merge_and_unload()
model.eval()
print("model ready")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

model ready


In [10]:
t0 = time.time()
BATCH = 8
i = 0
n_fail = 0
while i < len(todo):
    b = todo[i:i+BATCH]
    try:
        enc = tok([r["prompt"] for r in b], return_tensors="pt", padding=True,
                  truncation=True, max_length=4096).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for j, r in enumerate(b):
            txt = tok.decode(gen[j][enc["input_ids"].shape[1]:],
                             skip_special_tokens=True).split("\n")[0]
            append_record(OUT, dict(task=r["task"], sample=r["sample"],
                                    answer=r["answer"], raw=txt, error=None,
                                    model_version=CFG["name"], parsed=None))
        i += len(b)
    except Exception as e:
        n_fail += 1
        if BATCH > 1: BATCH = max(1, BATCH // 2)
        else:
            for r in b:
                append_record(OUT, dict(task=r["task"], sample=r["sample"],
                                        answer=r["answer"], raw="", error=repr(e),
                                        model_version=CFG["name"], parsed=None))
            i += len(b)
    if i % 100 < BATCH or i >= len(todo):
        print(f"  {i}/{len(todo)} (fail {n_fail}) {(time.time()-t0)/60:.1f} min", flush=True)

print(f"-> {OUT}: {len(todo)} processed, {n_fail} failed, {(time.time()-t0)/60:.1f} min")


  104/873 (fail 0) 1.0 min
  200/873 (fail 0) 1.8 min
  304/873 (fail 0) 2.8 min
  400/873 (fail 0) 3.6 min
  504/873 (fail 0) 4.6 min
  600/873 (fail 0) 5.5 min
  704/873 (fail 0) 6.4 min
  800/873 (fail 0) 7.3 min
  873/873 (fail 0) 8.0 min
-> /content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld-v6__MB-89xx.jsonl.gz: 873 processed, 0 failed, 8.0 min


Next: run the OTHER config (switch CFG), then locally:
`MASLD_LLM_ROOT=... python3 scripts/score_mb89xx.py`
The scorer appends leaderboard rows for the new archives automatically.
